# Music Streaming Habits Analyzer
* The Scenario: You are given three separate datasets: "User Demographics", "Track Metadata" (artist, genre, duration), and "Listening History" (who listened to what, and when).
* Combining Datasets: You must use pd.concat() to stitch together listening history from Q1 and Q2. Then, use pd.merge() to join the 'Listening History' table with the 'Track Metadata' table so you know the genre of the song played, acting like a SQL left join.
* Modifying DataFrames: You create a new calculated column called Minutes_Played by dividing the Milliseconds column by 60,000. Use .drop() to clear redundant system ID columns and .rename() poorly named columns.
* Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the time of day into "Morning", "Afternoon", "Evening" and "Night" based on the timestamp.
* Grouping and Aggregation: Using .groupby(), you group the data by 'Genre' and use .agg() to find the total minutes played, the average song duration, and the unique count of listeners for each genre.
* Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a heat-map-ready matrix showing "User Age Group" as rows, "Music Genre" as columns, and "Total Listens" as the values.


* Combining Datasets: You must use pd.concat() to stitch together listening history from Q1 and Q2. Then, use pd.merge() to join the 'Listening History' table with the 'Track Metadata' table so you know the genre of the song played, acting like a SQL left join.


In [139]:
import numpy as np
import pandas as pd
df_q1 = pd.read_csv('listening_history_q1 - listening_history_q1.csv')
df_q2 = pd.read_csv('listening_history_q2 - listening_history_q2.csv')
df_track =pd.read_csv('track_metadata - track_metadata.csv')
df_user_demographics = pd.read_csv('user_demographics - user_demographics.csv')

all_q = pd.concat([df_q1, df_q2] , ignore_index = True)
all_q = all_q.rename(columns = {'trk_ref_id': 'track_id'})
df = pd.merge(all_q, df_track , on= 'track_id', how= 'left')
df

,listen_id,usr_ref_id,track_id,timestamp,artist_name,Genre,length_in_ms,internal_db_id
0,1,167,146,2025-01-23 22:19:45,Artist_46,Electronic,121062,119870
1,3,90,105,2025-01-06 15:32:21,Artist_5,Electronic,285983,138102
2,5,39,119,2025-02-12 11:12:04,Artist_19,Hip-Hop,134397,625830
3,6,126,149,2025-03-05 19:52:42,Artist_49,Pop,272617,334677
4,8,173,101,2025-01-04 5:56:14,Artist_1,Pop,205999,897606
...,...,...,...,...,...,...,...,...
1495,1494,175,126,2025-06-15 7:30:32,Artist_26,Rock,298352,319930
1496,1495,165,101,2025-05-22 13:18:12,Artist_1,Pop,205999,897606
1497,1496,49,128,2025-06-13 22:29:31,Artist_28,Classical,267851,617313
1498,1497,120,137,2025-04-26 2:20:18,Artist_37,Hip-Hop,259407,257504


* Modifying DataFrames: You create a new calculated column called Minutes_Played by dividing the Milliseconds column by 60,000. Use .drop() to clear redundant system ID columns and .rename() poorly named columns.


In [140]:
df['Minutes_Played'] = df['length_in_ms']/60000
df = df.drop(['internal_db_id'], axis =1, errors = 'ignore')
df.rename(columns= {'listen_id': 'Listen id',
                    'usr_ref_id': 'user_id',
                    'track_id': 'Track ID',
                    'timestamp': 'Time Stamp'
                   }, inplace = True )
df = pd.merge(df_user_demographics, df, on= 'user_id', how= 'left')
df

,user_id,User Age Group,subscription_tier,sys_user_hash,Listen id,Track ID,Time Stamp,artist_name,Genre,length_in_ms,Minutes_Played
0,1,45-54,Premium,usr_1392,50,107,2025-03-19 3:37:20,Artist_7,Hip-Hop,283551,4.725850
1,1,45-54,Premium,usr_1392,236,141,2025-01-19 16:53:00,Artist_41,Electronic,262202,4.370033
2,1,45-54,Premium,usr_1392,566,120,2025-03-16 10:09:26,Artist_20,Rock,292627,4.877117
3,1,45-54,Premium,usr_1392,575,110,2025-01-21 3:01:47,Artist_10,Rock,287280,4.788000
4,1,45-54,Premium,usr_1392,1214,108,2025-01-23 8:54:19,Artist_8,Pop,181476,3.024600
...,...,...,...,...,...,...,...,...,...,...,...
1495,200,45-54,Free,usr_5484,60,125,2025-03-23 13:44:09,Artist_25,Rock,159954,2.665900
1496,200,45-54,Free,usr_5484,534,128,2025-03-19 6:57:27,Artist_28,Classical,267851,4.464183
1497,200,45-54,Free,usr_5484,13,128,2025-06-21 15:01:15,Artist_28,Classical,267851,4.464183
1498,200,45-54,Free,usr_5484,477,133,2025-05-30 14:23:27,Artist_33,Rock,171934,2.865567


* Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the time of day into "Morning", "Afternoon", "Evening" and "Night" based on the timestamp.


In [141]:
df.columns = df.columns.str.strip()
df[['Date', 'Time']]= df['Time Stamp'].str.split(' ', expand = True)
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.time
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour
def type_time(hour):
    if 5<= hour < 12:   
        
        return 'Morning'
    elif 12<= hour <17:
        return 'Afternoon'
    elif 17<= hour < 21 : 
        return 'Evening'
    else :
        return 'Night' 
        
df['Time Period']= df['Hour'].apply(type_time)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            1500 non-null   int64  
 1   User Age Group     1500 non-null   object 
 2   subscription_tier  1500 non-null   object 
 3   sys_user_hash      1500 non-null   object 
 4   Listen id          1500 non-null   int64  
 5   Track ID           1500 non-null   int64  
 6   Time Stamp         1500 non-null   object 
 7   artist_name        1500 non-null   object 
 8   Genre              1500 non-null   object 
 9   length_in_ms       1500 non-null   int64  
 10  Minutes_Played     1500 non-null   float64
 11  Date               1500 non-null   object 
 12  Time               1500 non-null   object 
 13  Hour               1500 non-null   int32  
 14  Time Period        1500 non-null   object 
dtypes: float64(1), int32(1), int64(4), object(9)
memory usage: 170.1+ KB


* Grouping and Aggregation: Using .groupby(), you group the data by 'Genre' and use .agg() to find the total minutes played, the average song duration, and the unique count of listeners for each genre.


In [145]:
analy_df = df.groupby('Genre').agg({
    'Minutes_Played': ['sum', 'mean'],
    'user_id': 'nunique'
})

# analy_df
df

,user_id,User Age Group,subscription_tier,sys_user_hash,Listen id,Track ID,Time Stamp,artist_name,Genre,length_in_ms,Minutes_Played,Date,Time,Hour,Time Period
0,1,45-54,Premium,usr_1392,50,107,2025-03-19 3:37:20,Artist_7,Hip-Hop,283551,4.725850,2025-03-19,03:37:20,3,Night
1,1,45-54,Premium,usr_1392,236,141,2025-01-19 16:53:00,Artist_41,Electronic,262202,4.370033,2025-01-19,16:53:00,16,Afternoon
2,1,45-54,Premium,usr_1392,566,120,2025-03-16 10:09:26,Artist_20,Rock,292627,4.877117,2025-03-16,10:09:26,10,Morning
3,1,45-54,Premium,usr_1392,575,110,2025-01-21 3:01:47,Artist_10,Rock,287280,4.788000,2025-01-21,03:01:47,3,Night
4,1,45-54,Premium,usr_1392,1214,108,2025-01-23 8:54:19,Artist_8,Pop,181476,3.024600,2025-01-23,08:54:19,8,Morning
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,200,45-54,Free,usr_5484,60,125,2025-03-23 13:44:09,Artist_25,Rock,159954,2.665900,2025-03-23,13:44:09,13,Afternoon
1496,200,45-54,Free,usr_5484,534,128,2025-03-19 6:57:27,Artist_28,Classical,267851,4.464183,2025-03-19,06:57:27,6,Morning
1497,200,45-54,Free,usr_5484,13,128,2025-06-21 15:01:15,Artist_28,Classical,267851,4.464183,2025-06-21,15:01:15,15,Afternoon
1498,200,45-54,Free,usr_5484,477,133,2025-05-30 14:23:27,Artist_33,Rock,171934,2.865567,2025-05-30,14:23:27,14,Afternoon


* Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a heat-map-ready matrix showing "User Age Group" as rows, "Music Genre" as columns, and "Total Listens" as the values.


In [146]:
df_heat_map_matrix = pd.pivot_table(
    df,
    values= 'Listen id',
    index = ['User Age Group'],
    columns = 'Genre',
    aggfunc = 'sum'
)
df_heat_map_matrix
                    

Genre,Classical,Electronic,Hip-Hop,Jazz,Pop,Rock
User Age Group,,,,,,
18-24,7844,56093,47142,21910,47109,53480
25-34,4724,50374,46133,23557,41294,48242
35-44,7608,45774,40140,24439,41796,53405
45-54,9014,68550,42800,31598,37591,75441
55+,7197,48535,41109,24585,37099,41167
